In [1]:
import Gmsh: gmsh
using Pkg
Pkg.add("MeshIO")
using MeshIO

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
Precompiling project...
   7379.8 ms  ✓ MeshIO
   9789.2 ms  ✓ Plots → FileIOExt
  50585.6 ms  ✓ JLD2
  31416.4 ms  ✓ Gridap
   6245.1 ms  ✓ GridapDistributed
   6188.4 ms  ✓ GridapGmsh
  6 dependencies successfully precompiled in 104 seconds. 299 already precompiled.


In [29]:
function check_mesh_2d(h; filename="check2d.msh")
    gmsh.initialize()
    gmsh.model.add("unit_square")
    gmsh.model.geo.addPoint(0,0,0,1,1) # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1,0,0,1,2)
    gmsh.model.geo.addPoint(1,1,0,1,3)
    gmsh.model.geo.addPoint(0,1,0,1,4)
    gmsh.model.geo.addLine(1,2,1)
    gmsh.model.geo.addLine(2,3,2) # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3,4,3)
    gmsh.model.geo.addLine(4,1,4)
        
    gmsh.model.geo.addCurveLoop([1,2,3,4],1)
        
    gmsh.model.geo.addPlaneSurface([1],1)

    N = Int(round(1/h))  # number of elements per edge

    gmsh.model.geo.mesh.setTransfiniteCurve(1, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(2, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(3, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(4, N+1)
    #triangulation
    gmsh.model.geo.mesh.setTransfiniteSurface(1)

    gmsh.model.geo.synchronize()

    gmsh.model.mesh.generate(2)

    gmsh.write(filename)
    gmsh.finalize()
end

check_mesh_2d (generic function with 1 method)

In [30]:
h = 0.05; # target mesh size
check_mesh_2d(h)
run(`python3 -m pip install --user meshio`)
run(`python3 -c "import meshio; m=meshio.read('check2d.msh'); meshio.write('check2d.vtu', m)"`)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00271596s, CPU 0s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Transfinite)
Info    : Done meshing 2D (Wall 0.00228815s, CPU 0.000961s)
Info    : 441 nodes 884 elements
Info    : Writing 'check2d.msh'...
Info    : Done writing 'check2d.msh'



Info: VTU format cannot write cell_sets. Converting them to cell_data...


Process(`python3 -c "import meshio; m=meshio.read('check2d.msh'); meshio.write('check2d.vtu', m)"`, ProcessExited(0))

In [4]:
function check_mesh_3d(h; filename="check3d.msh")
    gmsh.initialize()
    gmsh.model.add("unit_cube")

    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1)
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2)
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3)
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4)
    gmsh.model.geo.addLine(1, 2, 1)
    gmsh.model.geo.addLine(2, 3, 2)
    gmsh.model.geo.addLine(3, 4, 3)
    gmsh.model.geo.addLine(4, 1, 4)
    gmsh.model.geo.addCurveLoop([1,2,3,4], 1)
    gmsh.model.geo.addPlaneSurface([1], 1)
    # Extrude surface by dz = 1.0
    ext = gmsh.model.geo.extrude([(2, 1)], 0.0, 0.0, 1.0)
    gmsh.model.geo.synchronize()

    gmsh.model.mesh.generate(3)

    gmsh.write(filename)
    gmsh.finalize()

end

check_mesh_3d (generic function with 1 method)

In [5]:
h2 = 0.08; # target mesh size
check_mesh_3d(h2)
run(`python3 -m pip install --user -U meshio`)
run(`python3 -c "import meshio; m=meshio.read('check3d.msh'); m.cell_sets = {}; meshio.write('check3d.vtk', m)"`)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 60%] Meshing curve 9 (Line)
Info    : [ 70%] Meshing curve 11 (Line)
Info    : [ 80%] Meshing curve 12 (Line)
Info    : [ 90%] Meshing curve 16 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.000698268s, CPU 0.000555s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 13 (Surface, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 17 (Surface, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 21 (Surface, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 25 (Surface, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 26 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (W

component to gmsh:dim_tags.


Process(`python3 -c "import meshio; m=meshio.read('check3d.msh'); m.cell_sets = {}; meshio.write('check3d.vtk', m)"`, ProcessExited(0))